# Pattern Visualization Demo

This notebook demonstrates how to use the pattern visualization module to overlay ML-detected patterns on candlestick charts.

## Setup

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Image
import base64

# Add parent directory to path to make wave imports work
sys.path.append(str(Path(os.getcwd()).parent))

# Import Waveseer modules
from wave.patterns import PatternType, PatternMatch, detect_patterns
from wave.ml.viz.pattern_viz import PatternVisualizer, draw_patterns_on_chart, visualize_attention_map
from wave.ml.viz.chart_integration import draw_chart_with_patterns, augment_chart_with_ml_patterns

## Generate Sample Data

First, let's generate some sample price data with patterns for visualization.

In [ ]:
def generate_sample_ohlcv_data(n_periods=200, with_trends=True, seed=42):
    """Generate sample OHLCV data for testing."""
    np.random.seed(seed)
    dates = pd.date_range('2023-01-01', periods=n_periods)
    
    if with_trends:
        # Create more realistic price movements with trends
        base = 100
        # Create a more realistic price path with trends
        change_points = np.linspace(0, n_periods-1, 8).astype(int)  # 8 trend changes
        trends = np.random.normal(0, 0.5, len(change_points))  # Random trends
        
        # Create daily changes with trend influence
        daily_changes = np.random.normal(0, 1, n_periods)  # Base noise
        
        # Add trends to daily changes
        for i in range(len(change_points)-1):
            start, end = change_points[i], change_points[i+1]
            daily_changes[start:end] += trends[i]  # Add trend bias
        
        close = base + np.cumsum(daily_changes)
    else:
        # Simple random walk
        base = 100
        moves = np.cumsum(np.random.normal(0, 1, n_periods))
        close = base + moves
    
    # Create open, high, low based on close
    daily_volatility = 1.5
    high = close + np.random.uniform(0, daily_volatility, n_periods)
    low = close - np.random.uniform(0, daily_volatility, n_periods)
    open_prices = close - np.random.uniform(-daily_volatility/2, daily_volatility/2, n_periods)
    
    # Create volume - higher volume on big price moves
    base_volume = np.random.uniform(1000, 5000, n_periods)
    price_changes = np.abs(np.diff(np.append([0], close)))
    volume_factor = 1 + (price_changes / np.mean(price_changes))
    volume = base_volume * volume_factor
    
    df = pd.DataFrame({
        'open': open_prices,
        'high': high,
        'low': low,
        'close': close,
        'volume': volume
    }, index=dates)
    
    return df

# Generate sample data
df = generate_sample_ohlcv_data(n_periods=200, with_trends=True)

# Display first few rows
df.head()

## Create Sample Pattern Detections

Let's create some sample pattern detections to visualize.

In [ ]:
# Create sample pattern matches manually
sample_patterns = {
    PatternType.HEAD_AND_SHOULDERS: [
        PatternMatch(
            pattern_id="hs_1",
            pattern_type=PatternType.HEAD_AND_SHOULDERS,
            score=0.85,  # High confidence
            start_idx=20,
            end_idx=50,
            bars_matched=31,
            indicator_scores={"price": 0.9, "volume": 0.7}
        )
    ],
    PatternType.DOUBLE_BOTTOM: [
        PatternMatch(
            pattern_id="db_1",
            pattern_type=PatternType.DOUBLE_BOTTOM,
            score=0.68,  # Medium confidence
            start_idx=80,
            end_idx=110,
            bars_matched=31,
            indicator_scores={"price": 0.7, "volume": 0.6}
        )
    ],
    PatternType.RISING_WEDGE: [
        PatternMatch(
            pattern_id="rw_1",
            pattern_type=PatternType.RISING_WEDGE,
            score=0.42,  # Low confidence
            start_idx=140,
            end_idx=170,
            bars_matched=31,
            indicator_scores={"price": 0.4, "volume": 0.5}
        )
    ]
}

# Let's also run the built-in pattern detector for comparison
try:
    detected_patterns = detect_patterns(df)
    print(f"Detected {sum(len(patterns) for patterns in detected_patterns.values())} patterns")
except Exception as e:
    print(f"Built-in pattern detection not fully implemented: {e}")
    detected_patterns = {}

## Basic Pattern Visualization

Now let's visualize the patterns on a candlestick chart.

In [ ]:
# Helper function to display base64 images in the notebook
def display_b64_img(b64_str):
    """Display a base64 encoded image in the notebook."""
    if not b64_str.startswith('data:'):
        b64_str = f"data:image/png;base64,{b64_str}"
    display(HTML(f"<img src='{b64_str}' style='max-width:100%'>"))

# Draw chart with patterns using the PatternVisualizer
visualizer = PatternVisualizer()
img_str = visualizer.draw_patterns_on_chart(
    df=df, 
    patterns=sample_patterns,
    title="Sample Price Chart with Pattern Overlays"
)

# Display the chart
display_b64_img(img_str)

## Using the Chart Integration Module

The `chart_integration.py` module provides functions to integrate pattern visualization with existing charts.

In [ ]:
# Use the integrated chart function
img_str = draw_chart_with_patterns(
    df=df, 
    patterns=sample_patterns, 
    title="Chart with Integrated Pattern Visualization"
)

# Display the chart
display_b64_img(img_str)

## Simulating ML Model Predictions

Let's simulate ML model predictions and visualize them.

In [ ]:
# Simulate ML model predictions
ml_predictions = [
    {
        'pattern_type': PatternType.HEAD_AND_SHOULDERS,
        'start_idx': 20,
        'end_idx': 50,
        'confidence': 0.92,
        'metadata': {'model': 'transformer', 'importance_scores': [0.8, 0.7, 0.9]}
    },
    {
        'pattern_type': PatternType.DOUBLE_BOTTOM,
        'start_idx': 80,
        'end_idx': 110,
        'confidence': 0.75,
        'metadata': {'model': 'cnn', 'importance_scores': [0.6, 0.8, 0.7]}
    },
    {
        'pattern_type': PatternType.ASCENDING_TRIANGLE,
        'start_idx': 130,
        'end_idx': 160,
        'confidence': 0.45,  # Low confidence
        'metadata': {'model': 'lstm', 'importance_scores': [0.4, 0.5, 0.4]}
    }
]

# Visualize ML predictions with a confidence threshold
img_str = augment_chart_with_ml_patterns(
    df=df, 
    ml_predictions=ml_predictions,
    confidence_threshold=0.6,  # Only show patterns with confidence > 0.6
    title="ML Detected Patterns (Confidence > 60%)"
)

# Display the chart
display_b64_img(img_str)

## Visualizing Transformer Attention Maps

The pattern visualization module also includes functions to visualize transformer model attention maps.

In [ ]:
# Create a sample attention map
def generate_sample_attention(seq_len=200, n_heads=8):
    """Generate a sample attention map for visualization."""
    np.random.seed(42)
    
    # Create a sample attention matrix with some structure
    # In reality this would come from a transformer model
    
    # Create base random attention
    attn = np.random.rand(n_heads, seq_len, seq_len)
    
    # Add diagonal emphasis (self-attention)
    for h in range(n_heads):
        for i in range(seq_len):
            # Emphasize self-attention
            attn[h, i, i] += 0.5
            
            # Emphasize local context (nearby tokens)
            window = 5
            for j in range(max(0, i-window), min(seq_len, i+window+1)):
                attn[h, i, j] += 0.2 * (1 - abs(i-j)/window)
                
    # Add some pattern-specific attention for head 0
    # (simulate detecting a pattern around indices 40-60)
    pattern_start, pattern_end = 40, 60
    for i in range(pattern_start, pattern_end):
        for j in range(pattern_start, pattern_end):
            attn[0, i, j] += 0.3
    
    # Normalize each row to sum to 1 (as in softmax output)
    for h in range(n_heads):
        row_sums = attn[h].sum(axis=1, keepdims=True)
        attn[h] = attn[h] / row_sums
    
    return attn

# Generate sample attention map
attention_weights = generate_sample_attention(len(df), n_heads=8)

# Visualize attention map for head 0
img_str = visualize_attention_map(
    df=df,
    attention_weights=attention_weights,
    head_idx=0,
    title="Transformer Attention Map - Head 0"
)

# Display the attention map
display_b64_img(img_str)

## Customizing the Pattern Visualizer

You can customize the PatternVisualizer with custom colors and confidence thresholds.

In [ ]:
# Create a custom visualizer with different colors
custom_colors = {
    PatternType.HEAD_AND_SHOULDERS: "#8B0000",  # Dark red
    PatternType.DOUBLE_BOTTOM: "#006400",       # Dark green
    PatternType.RISING_WEDGE: "#00008B"         # Dark blue
}

custom_visualizer = PatternVisualizer(
    pattern_colors=custom_colors,
    default_alpha=0.25,  # More transparent
    confidence_thresholds={"low": 0.5, "medium": 0.8}  # Custom thresholds
)

# Draw chart with custom visualizer
img_str = custom_visualizer.draw_patterns_on_chart(
    df=df, 
    patterns=sample_patterns,
    title="Custom Pattern Visualization"
)

# Display the chart
display_b64_img(img_str)

## Conclusion

This notebook demonstrated how to use the pattern visualization module to:

1. Visualize ML-detected patterns on candlestick charts
2. Customize pattern visualizations with different colors and styles
3. Integrate pattern visualization with existing charts
4. Visualize transformer attention maps

This can help you understand and interpret the patterns detected by your ML models, and provide visual feedback to improve model performance.